In [ ]:
!pip install crewai==0.28.8 gradio==3.50.2 openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-cohere to determine which version is compatible with other requirements. This could take a while.
INFO: pip 

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "..."

In [ ]:
from crewai import Agent, Task, Crew
import gradio as gr

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:918: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warn(


In [ ]:
idea_generator = Agent(
    role="Startup Idea Generator",
    goal="Generate innovative AI startup ideas",
    backstory="Expert in identifying unique startup opportunities."
)

market_researcher = Agent(
    role="Market Researcher",
    goal="Analyze market demand and competition",
    backstory="Specialist in validating ideas using market trends."
)

business_strategist = Agent(
    role="Business Strategist",
    goal="Create business models and growth strategies",
    backstory="Expert in scaling startups profitably."
)

pitch_creator = Agent(
    role="Pitch Creator",
    goal="Create compelling startup pitches",
    backstory="Expert in storytelling and investor pitching."
)

In [ ]:
chat_history = []

def run_startup_bot(user_input):
    global chat_history

    try:
        chat_history.append(f"User: {user_input}")
        context = "\n".join(chat_history[-3:])  # limit memory

        # TASKS (short + fast)
        task1 = Task(
            description=f"Generate 2 AI startup ideas based on: {context}",
            expected_output="2 short ideas (max 30 words each)",
            agent=idea_generator
        )

        task2 = Task(
            description="Give target users, demand, competitors (1 line each)",
            expected_output="3 short bullet points",
            agent=market_researcher
        )

        task3 = Task(
            description="Give revenue model, pricing, growth (1 line each)",
            expected_output="3 short bullet points",
            agent=business_strategist
        )

        task4 = Task(
            description="Give pitch: problem, solution, vision (1 line each)",
            expected_output="3 short lines",
            agent=pitch_creator
        )

        crew = Crew(
            agents=[idea_generator, market_researcher, business_strategist, pitch_creator],
            tasks=[task1, task2, task3, task4]
        )

        result = crew.kickoff()

        # 🔥 IMPORTANT FIX
        result = str(result)

        chat_history.append(f"Bot: {result}")

        return result

    except Exception as e:
        return f"⚠️ Error: {str(e)}"

In [ ]:
def chatbot_interface(message, history):
    response = run_startup_bot(message)
    history = history + [(message, response)]
    return history, history

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🚀 AI Startup Idea Generator Chatbot")

    chatbot = gr.Chatbot(height=400)

    with gr.Row():
        msg = gr.Textbox(placeholder="Enter domain (e.g., fintech, healthcare)")
        send = gr.Button("Send 🚀")

    clear = gr.Button("Clear Chat ❌")
    state = gr.State([])

    send.click(chatbot_interface, [msg, state], [chatbot, state]).then(
        lambda: "", None, msg
    )

    msg.submit(chatbot_interface, [msg, state], [chatbot, state])

    clear.click(lambda: ([], []), outputs=[chatbot, state])

In [ ]:
demo.queue().launch()

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://a795efebbfff8b503e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
